# Lab 05 — Superdense Coding

Alice sends **two classical bits** to Bob by transmitting **one qubit**,
using a Bell pair they share in advance.

| Qubit | Role |
|---|---|
| `q[0]` | Alice's half of the Bell pair (encodes the message, then sent to Bob) |
| `q[1]` | Bob's half of the Bell pair |

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters
`MESSAGE = "dc"`: bit `d` switches on the Z gate, bit `c` switches on the X gate.

In [ ]:
MESSAGE = "11"   # one of "00", "01", "10", "11"
SHOTS = 1024
SEED = 7

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

from common.qlab import save_circuit

if MESSAGE not in ("00", "01", "10", "11"):
    raise ValueError("MESSAGE must be a 2-bit string: 00, 01, 10 or 11.")

backend = AerSimulator(seed_simulator=SEED)

## Build the circuit
Alice's encoding depends on the message, so the Z and X gates are added
in Python. To make the counts key read directly as the message `dc`,
`q[0]` is measured into `c[1]` and `q[1]` into `c[0]`.

In [ ]:
def superdense_circuit(message):
    d, c_bit = int(message[0]), int(message[1])

    q = QuantumRegister(2, "q")
    c = ClassicalRegister(2, "c")
    qc = QuantumCircuit(q, c)

    # Shared Bell pair
    qc.h(q[0])
    qc.cx(q[0], q[1])
    qc.barrier(label="Entanglement Preparation")

    # Alice encodes 2 bits on her qubit: Z^d X^c
    if c_bit:
        qc.x(q[0])
    if d:
        qc.z(q[0])
    qc.barrier(label="Alice Message Encoding")

    # Bob decodes with a Bell-basis measurement
    qc.cx(q[0], q[1])
    qc.h(q[0])
    qc.barrier(label="Bob Message Decoding")

    qc.measure(q[0], c[1])
    qc.measure(q[1], c[0])
    qc.barrier(label="Decoded Message")

    qc.reset(q[0])
    qc.reset(q[1])
    qc.barrier(label="Qubits Reset - Available for Reuse")

    return qc


qc = superdense_circuit(MESSAGE)

print("Z gate:", "ACTIVATED" if MESSAGE[0] == "1" else "NOT ACTIVATED")
print("X gate:", "ACTIVATED" if MESSAGE[1] == "1" else "NOT ACTIVATED")

save_circuit(qc, "circuit.png", scale=1.0);

## Run and decode

In [ ]:
counts = backend.run(transpile(qc, backend), shots=SHOTS).result().get_counts()
decoded = max(counts, key=counts.get)

print("Counts:", counts)
print("\nMessage sent:", MESSAGE)
print("Bob decoded :", decoded)
print("Correct     :", decoded == MESSAGE)

plot_histogram(counts, title=f"Message {MESSAGE}")

## Verification — all four messages

In [ ]:
for message in ("00", "01", "10", "11"):
    counts = backend.run(transpile(superdense_circuit(message), backend),
                         shots=SHOTS).result().get_counts()
    print(f"sent {message} -> received {counts}")
    assert counts == {message: SHOTS}, (message, counts)

print("\nAll four messages decoded correctly in every shot.")

In [ ]:
qc.draw(output="mpl", fold=-1, scale=1.0)